# model-clinic Treatment Guide

This notebook demonstrates the full treatment workflow:

1. Create a synthetically broken model
2. Diagnose and score it (before)
3. Apply conservative treatment
4. Re-diagnose and score (after)
5. Inspect the treatment manifest
6. Roll back a treatment if needed

**Requirements**: `pip install model-clinic`

All examples use synthetic models — no real checkpoints or GPU required.

In [ ]:
import torch
from model_clinic import (
    diagnose, prescribe, apply_treatment, rollback_treatment,
    compute_health_score, print_health_score,
    SYNTHETIC_MODELS, make_healthy_mlp,
    TreatmentManifest,
)

## 1. Create a Sick Model

We'll use the `norm-drift` preset — a model where LayerNorm weights have drifted from 1.0 during fine-tuning. This is one of the most common problems after SFT with a high learning rate.

In [ ]:
# Build a model with norm drift + dead neurons + heavy tails
# (a realistic combo after aggressive fine-tuning)
from model_clinic._synthetic import (
    make_norm_drift_model, make_dead_neuron_model, make_heavy_tails_model
)

def make_sft_damaged_model(hidden=256, layers=4):
    """Simulate a model damaged by high-LR SFT."""
    sd = make_healthy_mlp(hidden, layers)

    # Norm drift: common after SFT without norm clamping
    for i in range(layers):
        sd[f"layers.{i}.norm.weight"] = torch.ones(hidden) * 3.5

    # Dead neurons in early layers
    w = sd["layers.0.linear.weight"]
    n_dead = int(w.shape[0] * 0.25)  # 25% dead
    w[:n_dead] = 0

    # Heavy tails in attention-like weight
    w = sd["layers.1.linear.weight"]
    flat = w.flatten()
    outlier_idx = torch.randperm(flat.numel())[:max(1, flat.numel() // 50)]
    flat[outlier_idx] = torch.randn(outlier_idx.numel()) * 60.0
    sd["layers.1.linear.weight"] = flat.reshape(w.shape)

    return sd

sick_model = make_sft_damaged_model()
print(f"Created damaged model with {len(sick_model)} tensors")

## 2. Before: Diagnose and Score

In [ ]:
findings_before = diagnose(sick_model)
health_before = compute_health_score(findings_before)

print("=== BEFORE TREATMENT ===")
print_health_score(health_before)
print()

print(f"Total findings: {len(findings_before)}")
for f in findings_before:
    print(f"  [{f.severity:5s}] {f.condition:30s}  {f.param_name}")

## 3. Prescribe Treatments

`prescribe()` maps findings to specific treatment actions. In `conservative=True` mode, only `low`-risk fixes are prescribed. This is the safe default.

Available risk levels:
- **low** — reversible, well-understood, safe for production (norm resets, dead neuron reinit)
- **medium** — effective but may change model behavior (outlier clamping, scale adjustments)
- **high** — significant changes; only for severely broken models (full layer reinit)

In [ ]:
prescriptions = prescribe(findings_before, conservative=True)

print(f"Conservative prescriptions: {len(prescriptions)}")
print()
for rx in prescriptions:
    print(f"  Rx: {rx.name}")
    print(f"      Risk   : {rx.risk}")
    print(f"      Target : {rx.finding.param_name}")
    print(f"      Action : {rx.description}")
    if rx.explanation:
        print(f"      Why    : {rx.explanation[:120]}")
    print()

## 4. Apply Treatments

`apply_treatment()` modifies the state dict in-place and returns a `TreatmentResult`. It automatically saves a backup tensor for rollback.

We'll also use a `TreatmentManifest` to log what changed — useful for auditing and reproducibility.

In [ ]:
import copy

# Work on a copy so we can compare before/after
working_model = copy.deepcopy(sick_model)

# Track applied treatments for rollback
applied_results = []

print("Applying treatments...")
print()
for rx in prescriptions:
    result = apply_treatment(working_model, rx)
    applied_results.append(result)

    status = "OK  " if result.success else "FAIL"
    print(f"  [{status}] {rx.name}")
    print(f"         {result.description}")
    print()

## 5. After: Re-diagnose and Score

In [ ]:
findings_after = diagnose(working_model)
health_after = compute_health_score(findings_after)

print("=== AFTER TREATMENT ===")
print_health_score(health_after)
print()

# Summary comparison
delta = health_after.overall - health_before.overall
sign = "+" if delta >= 0 else ""
print(f"Score change  : {health_before.overall} -> {health_after.overall}  ({sign}{delta} pts)")
print(f"Grade change  : {health_before.grade} -> {health_after.grade}")
print(f"Findings      : {len(findings_before)} -> {len(findings_after)}")

In [ ]:
# Which findings were resolved?
before_conds = {(f.condition, f.param_name) for f in findings_before}
after_conds  = {(f.condition, f.param_name) for f in findings_after}

resolved = before_conds - after_conds
new_issues = after_conds - before_conds

print(f"Resolved ({len(resolved)}):")
for cond, param in sorted(resolved):
    print(f"  [FIXED] {cond} on {param}")

if new_issues:
    print(f"\nNew issues introduced ({len(new_issues)}):")
    for cond, param in sorted(new_issues):
        print(f"  [NEW]   {cond} on {param}")
else:
    print("\nNo new issues introduced.")

## 6. Treatment Manifest

The `TreatmentManifest` is an audit log of every change made. It records:
- Which parameters were modified
- Before/after statistics (mean, std, norm)
- Checksums for verification
- Timestamps

In [ ]:
manifest = TreatmentManifest()

# Manually record what we did (in the full pipeline this is automatic)
for result in applied_results:
    if result.success:
        manifest.record(
            param_name=result.prescription.finding.param_name,
            action=result.prescription.action,
            description=result.description,
        )

print("Treatment Manifest:")
print(f"  Entries: {len(manifest.entries)}")
print()
for entry in manifest.entries:
    print(f"  - {entry.get('action', '?')} on {entry.get('param_name', '?')}")
    print(f"    {entry.get('description', '')}")

## 7. Rolling Back a Treatment

Every `TreatmentResult` contains a backup of the original tensor. `rollback_treatment()` restores it. This is safe to call even after saving — it only affects the in-memory state dict.

In [ ]:
# Roll back only the first treatment (for demonstration)
if applied_results and applied_results[0].success:
    first = applied_results[0]
    param_name = first.prescription.finding.param_name

    before_norm = working_model[param_name].norm().item()
    rollback_treatment(working_model, first)
    after_norm = working_model[param_name].norm().item()

    print(f"Rolled back: {first.prescription.name}")
    print(f"  Param: {param_name}")
    print(f"  Norm before rollback: {before_norm:.4f}")
    print(f"  Norm after  rollback: {after_norm:.4f}")
    print("  (norm is back to the pre-treatment value)")

## 8. Treatment Pipeline (One-Shot)

For production use, `create_pipeline()` packages the full diagnose → prescribe → treat cycle into a single call. It returns a `PipelineResult` with before/after health scores.

In [ ]:
from model_clinic import create_pipeline, TreatmentPipeline

# Fresh sick model
pipeline_model = make_sft_damaged_model()

pipeline = create_pipeline([
    {"conservative": True},   # Step 1: safe fixes only
])

result = pipeline.run(pipeline_model)

print(f"Pipeline results:")
print(f"  Findings      : {len(result.findings)}")
print(f"  Prescriptions : {len(result.prescriptions)}")
print(f"  Treatments    : {len(result.treatments)}")
if result.health_before and result.health_after:
    print(f"  Score before  : {result.health_before.overall}/100")
    print(f"  Score after   : {result.health_after.overall}/100")

## 9. Save the Treated Model

Once you're happy with the treatment, save the state dict back to disk.

In [ ]:
import os
import tempfile
from model_clinic import save_state_dict

output_path = os.path.join(tempfile.gettempdir(), "treated_model.pt")
save_state_dict(pipeline_model, output_path)

print(f"Saved treated model to: {output_path}")
print()
print("CLI equivalent:")
print("  model-clinic treat sick_model.pt --conservative --save treated_model.pt")

## Summary: Treatment Workflow

```
state_dict = load_state_dict("checkpoint.pt")

# 1. Diagnose
findings = diagnose(state_dict)

# 2. Score
health_before = compute_health_score(findings)

# 3. Prescribe (conservative = safe fixes only)
prescriptions = prescribe(findings, conservative=True)

# 4. Apply (each call modifies state_dict in-place, stores backup)
results = [apply_treatment(state_dict, rx) for rx in prescriptions]

# 5. Re-score
health_after = compute_health_score(diagnose(state_dict))

# 6. Save (or roll back if score dropped)
if health_after.overall >= health_before.overall:
    save_state_dict(state_dict, "treated.pt")
else:
    for r in results:
        rollback_treatment(state_dict, r)
```

## Next Steps

- **Notebook 03** — Training Monitor: catch problems before they become post-hoc repairs
- `docs/treatment_recipes.md` — Proven fix combinations for specific scenarios
- CLI: `model-clinic treat model.pt --dry-run` to preview without applying